# Explainable CNN for Potato Leaf Disease Detection

This notebook trains and evaluates the lightweight custom CNN used by the application. It is intentionally focused on one reproducible pipeline rather than several competing experiments.

The model classifies seven potato leaf categories: Bacteria, Fungi, Healthy, Nematode, Pest, Phytopthora, and Virus. The preprocessing, class order, model input shape, and exported model artifact are kept compatible with the FastAPI inference service.

## Pipeline

1. Load and audit the dataset.
2. Create a stratified train/validation/test split.
3. Apply training augmentation and shared evaluation preprocessing.
4. Train the custom CNN with checkpointing and early stopping.
5. Evaluate with accuracy, a classification report, and a confusion matrix.
6. Generate a prediction smoke test and Grad-CAM examples.
7. Export the model and the metadata required to serve it safely.

> Dataset and deployment limitations are recorded in the final section.

In [ ]:
# Imports and reproducibility
import json
import os
import random
from pathlib import Path

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass

# Avoid a TensorFlow Grappler layout-optimizer issue with deterministic GPU dropout.
tf.config.optimizer.set_experimental_options({'layout_optimizer': False})

GPU_DEVICES = tf.config.list_physical_devices('GPU')
for gpu in GPU_DEVICES:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
strategy = tf.distribute.MirroredStrategy()
GPU_NAMES = [device.name for device in GPU_DEVICES]

sns.set_theme(style='whitegrid')
print(f'TensorFlow: {tf.__version__}')
print(f'Keras: {keras.__version__}')
print(f'Visible GPUs: {GPU_NAMES}')
print(f'Replicas in sync: {strategy.num_replicas_in_sync}')

## 1. Configuration and Dataset Contract

Set `DATASET_DIR` when needed. On Kaggle, the notebook searches `/kaggle/input` for the directory containing the seven potato class folders, so the Kaggle dataset slug does not need to be hardcoded.

The explicit `CLASS_NAMES` order is part of the model contract. It must not be inferred from filesystem ordering.

In [ ]:
# Paths and model contract
IMG_SIZE = (224, 224)
CHANNELS = 3
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '32'))
EPOCHS = int(os.getenv('EPOCHS', '20'))
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

CLASS_NAMES = [
    'Bacteria',
    'Fungi',
    'Healthy',
    'Nematode',
    'Pest',
    'Phytopthora',
    'Virus',
]
DISPLAY_NAMES = {
    'Bacteria': 'Bacterial Disease',
    'Fungi': 'Fungal Disease',
    'Healthy': 'Healthy',
    'Nematode': 'Nematode Damage',
    'Pest': 'Pest Damage',
     'Phytopthora': 'Phytophthora',
    'Virus': 'Viral Disease',
}
EXPECTED_CLASS_INDICES = {name: index for index, name in enumerate(CLASS_NAMES)}

def has_expected_class_folders(path):
    return path.is_dir() and all((path / name).is_dir() for name in CLASS_NAMES)


def find_dataset_root():
    configured_path = os.getenv('DATASET_DIR')
    if configured_path:
        configured = Path(configured_path).expanduser()
        if not has_expected_class_folders(configured):
            raise FileNotFoundError(
                f'DATASET_DIR does not contain the expected class folders: {configured}'
            )
        return configured

    candidates = [Path('dataset'), Path('data/PlantVillage')]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        candidates = [kaggle_input, *sorted(kaggle_input.rglob('*')), *candidates]

    for candidate in candidates:
        if has_expected_class_folders(candidate):
            return candidate

    raise FileNotFoundError(
        'Could not find a directory containing all expected class folders. '
        f'Expected: {CLASS_NAMES}. Set DATASET_DIR explicitly if needed.'
    )


DATASET_DIR = find_dataset_root()
default_output_dir = (
    Path('/kaggle/working/artifacts')
    if Path('/kaggle/working').is_dir()
    else Path('artifacts')
)
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', str(default_output_dir))).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / 'explainable_cnn_model.keras'

print(f'Dataset: {DATASET_DIR.resolve()}')
print(f'Output:  {OUTPUT_DIR.resolve()}')
print(f'Classes: {CLASS_NAMES}')
print(f'Input:   {IMG_SIZE + (CHANNELS,)}')

In [ ]:
def build_image_dataframe(dataset_dir):
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f'Dataset directory does not exist: {dataset_dir}. '
            'Set DATASET_DIR to the directory containing the class folders.'
        )

    rows = []
    ignored_classes = []
    for class_dir in sorted(path for path in dataset_dir.iterdir() if path.is_dir()):
        if class_dir.name not in CLASS_NAMES:
            ignored_classes.append(class_dir.name)
            continue
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({'filepath': str(image_path.resolve()), 'label': class_dir.name})

    data = pd.DataFrame(rows, columns=['filepath', 'label'])
    if data.empty:
        raise ValueError(
            f'No supported images found in {dataset_dir}. '
            f'Expected class folders: {CLASS_NAMES}'
        )

    missing_classes = sorted(set(CLASS_NAMES) - set(data['label']))
    if missing_classes:
        raise ValueError(f'Missing expected class folders: {missing_classes}')
    if ignored_classes:
        print(f'Ignoring non-potato class folders: {ignored_classes}')

    return data.sort_values(['label', 'filepath']).reset_index(drop=True)


data = build_image_dataframe(DATASET_DIR)
counts = data['label'].value_counts().reindex(CLASS_NAMES)
print(f'Total images: {len(data):,}')
display(counts.rename('images').to_frame())

## 2. Dataset Audit

This is a compact audit: class balance and one representative image per class. The detailed RGB histogram exploration from the original notebook was removed because it did not affect preprocessing, architecture, or evaluation.

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 4))
for axis, class_name in zip(axes, CLASS_NAMES):
    sample = data[data['label'] == class_name].sample(1, random_state=SEED).iloc[0]
    with PILImage.open(sample['filepath']) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(f"{DISPLAY_NAMES[class_name]}\n({counts[class_name]:,} images)")
    axis.axis('off')
fig.suptitle('Representative samples by class')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'dataset_samples.png', dpi=160, bbox_inches='tight')
plt.show()

## 3. Stratified Train, Validation, and Test Split

The split is 80/10/10. Stratification preserves class proportions in every set, while the filepath checks prevent accidental overlap between sets.

In [ ]:
train_data, temporary_data = train_test_split(
    data,
    test_size=0.20,
    stratify=data['label'],
    random_state=SEED,
)
valid_data, test_data = train_test_split(
    temporary_data,
    test_size=0.50,
    stratify=temporary_data['label'],
    random_state=SEED,
)

train_data = train_data.reset_index(drop=True)
valid_data = valid_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

train_paths = set(train_data['filepath'])
valid_paths = set(valid_data['filepath'])
test_paths = set(test_data['filepath'])
assert train_paths.isdisjoint(valid_paths)
assert train_paths.isdisjoint(test_paths)
assert valid_paths.isdisjoint(test_paths)

split_summary = pd.DataFrame(
    {
        'train': train_data['label'].value_counts().reindex(CLASS_NAMES),
        'validation': valid_data['label'].value_counts().reindex(CLASS_NAMES),
        'test': test_data['label'].value_counts().reindex(CLASS_NAMES),
    }
).fillna(0).astype(int)
display(split_summary)
print(f'Train: {len(train_data):,} | Validation: {len(valid_data):,} | Test: {len(test_data):,}')

## 4. Preprocessing and Data Generators

The custom CNN receives RGB images resized to `224x224` and scaled from `[0, 255]` to `[0, 1]`. Training augmentation is applied only to the training set. Validation and test images use rescaling only.

The explicit `classes` argument guarantees that generator indices match the FastAPI class order.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
)
evaluation_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
if BATCH_SIZE % strategy.num_replicas_in_sync != 0:
    raise ValueError(
        'BATCH_SIZE must be divisible by the number of synchronized replicas.'
    )

generator_options = {
    'x_col': 'filepath',
    'y_col': 'label',
    'classes': CLASS_NAMES,
    'target_size': IMG_SIZE,
    'color_mode': 'rgb',
    'class_mode': 'categorical',
    'batch_size': BATCH_SIZE,
    'validate_filenames': True,
}

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_data, shuffle=True, seed=SEED, **generator_options
)
valid_gen = evaluation_datagen.flow_from_dataframe(
    dataframe=valid_data, shuffle=False, **generator_options
)
test_gen = evaluation_datagen.flow_from_dataframe(
    dataframe=test_data, shuffle=False, **generator_options
)

assert train_gen.class_indices == EXPECTED_CLASS_INDICES
batch_images, batch_labels = train_gen[0]
assert batch_images.shape[1:] == IMG_SIZE + (CHANNELS,)
assert batch_labels.shape[1] == len(CLASS_NAMES)
print(f'Batch images: {batch_images.shape}')
print(f'Batch labels: {batch_labels.shape}')
print(f'Class mapping: {train_gen.class_indices}')

## 5. Custom CNN Architecture

This is the lightweight model deployed by the application. Three convolutional blocks learn increasingly abstract visual features. Global average pooling keeps the classification head small compared with flattening the full feature map, while the final layer expands to the seven dataset classes.

In [ ]:
def build_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape, name='image')

    x = layers.Conv2D(32, 3, padding='same', name='conv_1')(inputs)
    x = layers.BatchNormalization(name='batch_norm_1')(x)
    x = layers.Activation('swish', name='swish_1')(x)
    x = layers.Conv2D(64, 3, padding='same', name='conv_2')(x)
    x = layers.BatchNormalization(name='batch_norm_2')(x)
    x = layers.Activation('swish', name='swish_2')(x)
    x = layers.MaxPooling2D(2, name='pool_1')(x)
    x = layers.Dropout(0.25, name='dropout_1')(x)

    x = layers.Conv2D(64, 3, padding='same', name='conv_3')(x)
    x = layers.BatchNormalization(name='batch_norm_3')(x)
    x = layers.Activation('swish', name='swish_3')(x)
    x = layers.Conv2D(128, 3, padding='same', name='conv_4')(x)
    x = layers.BatchNormalization(name='batch_norm_4')(x)
    x = layers.Activation('swish', name='swish_4')(x)
    x = layers.MaxPooling2D(2, name='pool_2')(x)
    x = layers.Dropout(0.25, name='dropout_2')(x)

    x = layers.Conv2D(128, 3, padding='same', name='conv_5')(x)
    x = layers.BatchNormalization(name='batch_norm_5')(x)
    x = layers.Activation('swish', name='swish_5')(x)
    x = layers.Conv2D(256, 3, padding='same', name='conv_6')(x)
    x = layers.BatchNormalization(name='batch_norm_6')(x)
    x = layers.Activation('swish', name='swish_6')(x)
    x = layers.MaxPooling2D(2, name='pool_3')(x)
    x = layers.Dropout(0.30, name='dropout_3')(x)

    x = layers.GlobalAveragePooling2D(name='global_average_pool')(x)
    x = layers.Dense(256, name='dense_features')(x)
    x = layers.BatchNormalization(name='batch_norm_dense')(x)
    x = layers.Activation('swish', name='swish_dense')(x)
    x = layers.Dropout(0.50, name='dropout_dense')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    return keras.Model(inputs=inputs, outputs=outputs, name='potato_disease_cnn')

with strategy.scope():
    model = build_model(IMG_SIZE + (CHANNELS,), len(CLASS_NAMES))
model.summary()

## 6. Compile and Train

Label smoothing reduces overconfident predictions. Training uses class weights computed from the training split so the smaller classes, especially Nematode, are not ignored. The best validation-loss checkpoint is exported, while early stopping prevents unnecessary training once validation performance stops improving.

In [ ]:
train_class_counts = train_data['label'].value_counts().reindex(CLASS_NAMES)
CLASS_WEIGHTS = {
    class_index: float(
        len(train_data) / (len(CLASS_NAMES) * int(class_count))
    )
    for class_index, class_count in enumerate(train_class_counts)
}
print(f'Train class counts: {train_class_counts.to_dict()}')
print(f'Class weights: {CLASS_WEIGHTS}')

with strategy.scope():
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=[keras.metrics.CategoricalAccuracy(name='accuracy')],
    )

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=BEST_MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1
    ),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=valid_gen,
    callbacks=callbacks,
    class_weight=CLASS_WEIGHTS,
)

## 7. Training History

In [ ]:
history_frame = pd.DataFrame(history.history)
epochs_completed = range(1, len(history_frame) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_completed, history_frame['accuracy'], label='Train')
axes[0].plot(epochs_completed, history_frame['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_completed, history_frame['loss'], label='Train')
axes[1].plot(epochs_completed, history_frame['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

fig.suptitle('Custom CNN training history')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Test Evaluation

The test set is evaluated only after selecting the best validation checkpoint. The confusion matrix and per-class metrics reveal errors that a single accuracy value can hide.

In [ ]:
best_model = keras.models.load_model(BEST_MODEL_PATH)
test_metrics = best_model.evaluate(test_gen, return_dict=True, verbose=1)
test_gen.reset()
probabilities = best_model.predict(test_gen, verbose=1)
predicted_indices = np.argmax(probabilities, axis=1)
true_indices = test_gen.classes

assert probabilities.shape == (len(test_data), len(CLASS_NAMES))
report = classification_report(
    true_indices,
    predicted_indices,
    labels=range(len(CLASS_NAMES)),
    target_names=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    zero_division=0,
)
report_dict = classification_report(
    true_indices,
    predicted_indices,
    labels=range(len(CLASS_NAMES)),
    target_names=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    output_dict=True,
    zero_division=0,
)
print(f"Test loss: {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy'] * 100:.2f}%")
print(report)

matrix = confusion_matrix(true_indices, predicted_indices, labels=range(len(CLASS_NAMES)))
fig, axis = plt.subplots(figsize=(7, 6))
sns.heatmap(
    matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    yticklabels=[DISPLAY_NAMES[name] for name in CLASS_NAMES],
    ax=axis,
)
axis.set_xlabel('Predicted label')
axis.set_ylabel('True label')
axis.set_title('Custom CNN confusion matrix')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160, bbox_inches='tight')
plt.show()

evaluation_artifact = {
    'test_loss': float(test_metrics['loss']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_samples': int(len(test_data)),
    'classification_report': report_dict,
}
(OUTPUT_DIR / 'evaluation.json').write_text(json.dumps(evaluation_artifact, indent=2))

## 9. Inference Smoke Test

This uses the same resize and `/255` preprocessing as the backend. It is a quick contract check before generating explanations or handing the artifact to the application.

In [ ]:
def prepare_image(filepath):
    image = keras.utils.load_img(filepath, target_size=IMG_SIZE, color_mode='rgb')
    array = keras.utils.img_to_array(image).astype('float32') / 255.0
    return image, np.expand_dims(array, axis=0)


def predict_filepath(filepath, classifier):
    image, batch = prepare_image(filepath)
    probabilities = classifier(batch, training=False).numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    return image, predicted_index, probabilities


sample_row = test_data.sample(1, random_state=SEED).iloc[0]
sample_image, sample_index, sample_probabilities = predict_filepath(
    sample_row['filepath'], best_model
)
print(f"True:      {DISPLAY_NAMES[sample_row['label']]}")
print(f"Predicted: {DISPLAY_NAMES[CLASS_NAMES[sample_index]]}")
print(f"Confidence: {sample_probabilities[sample_index] * 100:.2f}%")
assert len(sample_probabilities) == len(CLASS_NAMES)

plt.figure(figsize=(5, 5))
plt.imshow(sample_image)
plt.title(
    f"True: {DISPLAY_NAMES[sample_row['label']]} | "
    f"Predicted: {DISPLAY_NAMES[CLASS_NAMES[sample_index]]}"
)
plt.axis('off')
plt.show()

## 10. Grad-CAM Explainability

Grad-CAM visualizes the regions that contributed to a selected class score. It is an attention explanation, not a segmentation mask and not a validated percentage of infected tissue. The final convolutional layer is selected from the same model used for prediction, so a second architecture definition is unnecessary.

In [ ]:
def find_last_conv_layer(classifier):
    for layer in reversed(classifier.layers):
        if isinstance(layer, layers.Conv2D):
            return layer
    raise ValueError('No Conv2D layer found.')


def make_gradcam_heatmap(image_batch, classifier, class_index=None, target_layer=None):
    if target_layer is None:
        target_layer = find_last_conv_layer(classifier)
    grad_model = keras.Model(
        inputs=classifier.inputs,
        outputs=[target_layer.output, classifier.output],
    )

    with tf.GradientTape() as tape:
        convolution_output, predictions = grad_model({'image': image_batch}, training=False)
        if class_index is None:
            class_index = int(tf.argmax(predictions[0]))
        class_score = predictions[:, class_index]

    gradients = tape.gradient(class_score, convolution_output)
    if gradients is None:
        raise ValueError('Gradients were not produced for the selected layer.')

    weights = tf.reduce_mean(gradients, axis=(1, 2))
    feature_map = convolution_output[0]
    pooled_weights = weights[0]
    heatmap = tf.reduce_sum(feature_map * pooled_weights, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    maximum = tf.reduce_max(heatmap)
    heatmap = tf.where(maximum > 0, heatmap / maximum, tf.zeros_like(heatmap))
    return heatmap.numpy(), target_layer.name


def make_overlay(image, heatmap, alpha=0.45):
    original = np.asarray(image.convert('RGB').resize(IMG_SIZE), dtype='float32') / 255.0
    resized_heatmap = PILImage.fromarray(
        np.uint8(np.clip(heatmap, 0, 1) * 255)
    ).resize(IMG_SIZE, resample=PILImage.Resampling.BILINEAR)
    heatmap_array = np.asarray(resized_heatmap, dtype='float32') / 255.0
    colors = plt.get_cmap('jet')(np.uint8(heatmap_array * 255))[..., :3]
    return np.clip((1 - alpha) * original + alpha * colors, 0, 1)


target_layer = find_last_conv_layer(best_model)
print(f'Grad-CAM target layer: {target_layer.name}')

In [ ]:
number_of_examples = min(3, len(test_data))
example_rows = test_data.sample(number_of_examples, random_state=SEED)
fig, axes = plt.subplots(number_of_examples, 3, figsize=(12, 4 * number_of_examples), squeeze=False)

for row_index, (_, row) in enumerate(example_rows.iterrows()):
    image, image_batch = prepare_image(row['filepath'])
    probabilities = best_model(image_batch, training=False).numpy()[0]
    predicted_index = int(np.argmax(probabilities))
    heatmap, layer_name = make_gradcam_heatmap(
        image_batch, best_model, class_index=predicted_index, target_layer=target_layer
    )
    overlay = make_overlay(image, heatmap)
    predicted_name = DISPLAY_NAMES[CLASS_NAMES[predicted_index]]
    true_name = DISPLAY_NAMES[row['label']]

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f'True: {true_name}')
    axes[row_index, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[row_index, 1].set_title(f'Heatmap: {layer_name}')
    axes[row_index, 2].imshow(overlay)
    axes[row_index, 2].set_title(f'Predicted: {predicted_name} ({probabilities[predicted_index] * 100:.1f}%)')
    for axis in axes[row_index]:
        axis.axis('off')

fig.suptitle('Grad-CAM examples')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'gradcam_examples.png', dpi=160, bbox_inches='tight')
plt.show()

## 11. Export Model and Serving Metadata

The model is saved as a standalone `.keras` artifact. The JSON metadata records the preprocessing and class contract that the FastAPI service must follow.

For the application, configure `MODEL_PATH` to this exported model or copy it to the repository location expected by the backend.

In [ ]:
metadata_artifact = {
    'model_filename': BEST_MODEL_PATH.name,
    'architecture': 'lightweight custom CNN',
    'input_size': list(IMG_SIZE),
    'channels': CHANNELS,
    'color_mode': 'RGB',
    'rescaling': 'pixel / 255.0',
    'class_names': CLASS_NAMES,
    'class_indices': EXPECTED_CLASS_INDICES,
    'seed': SEED,
    'split': {'train': 0.8, 'validation': 0.1, 'test': 0.1},
    'dataset_dir': str(DATASET_DIR.resolve()),
    'dataset_images': int(len(data)),
    'tensorflow_version': tf.__version__,
    'gpu_devices': GPU_NAMES,
    'replicas_in_sync': int(strategy.num_replicas_in_sync),
    'batch_size': BATCH_SIZE,
    'epochs_requested': EPOCHS,
    'epochs_completed': len(history.history['loss']),
    'class_weights': {str(index): weight for index, weight in CLASS_WEIGHTS.items()},
    'train_class_counts': {
        name: int(count) for name, count in train_class_counts.items()
    },
    'layout_optimizer_disabled': True,
}

(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata_artifact, indent=2))
(OUTPUT_DIR / 'class_names.json').write_text(json.dumps(CLASS_NAMES, indent=2))
with (OUTPUT_DIR / 'model_summary.txt').open('w') as summary_file:
    best_model.summary(print_fn=lambda line: summary_file.write(line + '\n'))

print('Exported artifacts:')
for artifact in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {artifact.name}')

## Notes for Reuse

- Run the notebook on Kaggle with a GPU accelerator, or set `DATASET_DIR` and `OUTPUT_DIR` explicitly in another environment.
- A random image split is appropriate for reproducing the existing experiment, but a future study should consider grouping near-duplicate source images before splitting.
- Test metrics describe this dataset split and should not be treated as guaranteed field performance or a definitive agricultural diagnosis.
- Class weights are computed from the training split only; validation and test metrics remain unweighted.
- Transfer-learning benchmarks were removed from this operational notebook. If they are still needed for a paper, keep them in a separate comparison notebook.